In [2]:
import pandas as pd
import numpy as np

In [3]:
jobs_csv= pd.read_csv("../Data/dataset_rapid-linkedin-scraper_2026-11-07_04-05-56-981.csv")
jobs_csv.head()

,company_id,company_logo_url,company_name,company_url,easy_apply,employment_type,hiring_badge,industries,is_actively_hiring,is_early_applicant,...,job_id,job_title,job_url,location,num_applicants,posted_date,salary_range,search_keyword,seniority_level,time_posted
0,3276723,https://media.licdn.com/dms/image/v2/D560BAQHN...,Sii Poland,https://pl.linkedin.com/company/sii-polska,False,Full-time,Be an early applicant,IT Services and IT Consulting,False,True,...,4484344125,Senior Data Scientist – AdTech & Healthcare (f...,https://www.linkedin.com/jobs/view/4484344125,"Gdańsk, Pomorskie, Poland",Be among the first 25 applicants,2026-11-02,NaN,Data Scientist,Mid-Senior level,4 days ago
1,24189,https://media.licdn.com/dms/image/v2/C4D0BAQF6...,Rumos,https://pt.linkedin.com/company/rumos,True,Full-time,Be an early applicant,"IT Services and IT Consulting and Technology, ...",False,True,...,4486235459,Senior Data Scientist,https://www.linkedin.com/jobs/view/4486235459,Porto Metropolitan Area,Be among the first 25 applicants,2026-11-06,NaN,Data Scientist,Mid-Senior level,16 hours ago
2,1373602,https://media.licdn.com/dms/image/v2/C4D0BAQH3...,Hopper,https://ca.linkedin.com/company/hopper,False,Full-time,Be an early applicant,Software Development,False,True,...,4486551073,Scientifique de données principal(e)// Senior ...,https://www.linkedin.com/jobs/view/4486551073,"Québec, Quebec, Canada",Be among the first 25 applicants,2026-11-04,NaN,Data Scientist,Not Applicable,2 days ago
3,3276723,https://media.licdn.com/dms/image/v2/D560BAQHN...,Sii Poland,https://pl.linkedin.com/company/sii-polska,False,Full-time,Be an early applicant,IT Services and IT Consulting,False,True,...,4484328756,Senior Data Scientist – AdTech & Healthcare (f...,https://www.linkedin.com/jobs/view/4484328756,"Wrocław, Dolnośląskie, Poland",Be among the first 25 applicants,2026-11-02,NaN,Data Scientist,Mid-Senior level,4 days ago
4,1403,https://media.licdn.com/dms/image/v2/D4E0BAQFX...,Booz Allen Hamilton,https://www.linkedin.com/company/booz-allen-ha...,False,Full-time,Be an early applicant,Software Development,False,True,...,4487787319,Data Scientist,https://www.linkedin.com/jobs/view/4487787319,"Arlington, VA",Be among the first 25 applicants,2026-11-05,NaN,Data Scientist,Not Applicable,1 day ago


In [4]:
print(jobs_csv["job_description"].iloc[0])

Tired of building yet another chatbot POC? Do you want to train ML models that bring real monetary value and run at a billion-scale? Do you feel like introducing a brand-new probabilistic methodology for a real-world problem?We build the AI-powered operating system for healthcare marketing - programmatic advertising, data intelligence, patient affordability and AI-enabled engagement for the life sciences industry.We are looking for a person who knows how to design and implement statistical methods to extract signals from large, messy, real-world datasets and use them for prediction. Everything we build on top of that, from audience reach to specialty insight to measurement, is only as good as your layer. Your tasks Measuring existing model performance and finding ways to improve itOwning the design of the new methodology and training the next-gen model for increased predictive power, sample efficiency, and scalability to large datasetsExtracting unique entities and sparse attributes fr

In [5]:
from langdetect import detect

def detect_language(text):
    try:
        return detect(text)
    except Exception as e:
        return f"Error detecting language: {e}"

In [6]:
from pathlib import Path

non_english = []

for des in jobs_csv["job_description"].dropna().astype(str):
    language = detect_language(des)

    if language.startswith("Error detecting language:"):
        continue

    if language != "en":
        non_english.append(des)

output_path = Path("..") / "Data" / "job_descriptions.txt"
output_path.write_text("\n\n".join(non_english), encoding="utf-8")

294365

In [7]:
import re                                 
from pathlib import Path                                                                       
                                                                                                                    
file_path = Path("..") / "Data" / "translated_job_descriptions.txt"                                              
with open(file_path, "r", encoding="utf-8") as file:                                                             
    raw_paragraphs = [p.strip() for p in file.read().split("\n\n") if p.strip()]                                   
                                                                                                                    
cleaned_translations = [                                                                                         
    re.sub(r"\[.*?\]\s*", "", p).strip() for p in raw_paragraphs                                                 
]                                                                                                                
translation_iter = iter(cleaned_translations)                                                                    
                                                                                                                    
for idx, row in jobs_csv.iterrows():                                                                             
    des = row["job_description"]                                                                                   
    if pd.isna(des):                                                                                               
        continue                                                                                                     
    lang = detect_language(str(des))                                                                               
    if lang.startswith("Error detecting language:"):                                                               
        continue                                                                                                     
                                                                                                                    
    if lang != "en":                                                                                               
        try:                                                                                                         
            jobs_csv.at[idx, "job_description"] = next(translation_iter)                                               
        except StopIteration:                                                                                        
            print("Warning: Reached the end of translated descriptions.")                                              
            break                                                                                                      
                                                                                                                    
print("All non-English descriptions have been successfully replaced!") 

All non-English descriptions have been successfully replaced!


In [8]:
jobs_csv['job_description'][1]

'Rumos Consulting is recruiting a Senior Data Scientist for Porto (hybrid model). Key duties include exploring, analyzing, and evaluating structured and unstructured internal and external data to support business needs; executing the data valuation lifecycle (selection, collection, quality analysis, remediation, preparation, and Data Science techniques); developing predictive and prescriptive analyses using Machine Learning and Deep Learning algorithms along with dynamic visualization and data storytelling; applying advanced statistical techniques, text mining, geolocation, and Big Data extraction; designing Datamarts and Data Warehouses; ensuring end-to-end data quality and integration across business systems; and translating complex business challenges into mathematical/statistical models. Requirements: Higher education in Information Technology or a related field; a minimum of 9 years of professional experience as a Data Scientist; hands-on expertise with structured/unstructured dat

In [20]:
import re
import pandas as pd

def get_job_sentences(row, column="job_description"):
    description = row[column]

    if pd.isna(description):
        return []

    sentences = re.split(r"(?<=[.!?])\s+", str(description).strip())
    return [sentence for sentence in sentences if sentence]

In [23]:
sentences = get_job_sentences(jobs_csv.iloc[0])
print(sentences[5])  # Prints the second sentence

JOB-AV29DSii ensures that all hiring decisions are made solely on the basis of qualifications and competence.
